# 🎯 [OPS02_M05] 센서별 오탐 가중치 반영 및 항적 통합 신뢰도 지수 산출 엔진 개발
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS02/ops02_m05_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS02_M05 |
| **미션 주제** | 센서별 오탐 가중치 반영 및 항적 통합 신뢰도 지수 산출 엔진 개발 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 간 무력 충돌 사례에서는 드론·미사일이 여러 방향에서 동시에 들어오면서 레이더, 광학(EO/IR), 음향 센서가 같은 비행체를 각자 경보해 상황판에 중복 항적이 쏟아지고, 이 때문에 위협 수를 잘못 세거나 대응이 늦어지는 문제가 주목받았다. 본 프로젝트에서는 가상 방어 지역에 배치한 가상 센서 4기와 가상 비행체 궤적으로 교육용 경보 데이터를 코드로 만들어, 연동 규격과 통합 스키마 설계, 시각 동기화와 ENU 좌표 변환 전처리, 최단거리 매칭 Baseline, 확장 칼만 필터(EKF) 항적 추적·통합, 융합 신뢰도 지수, 전술 상황판(COP), 조작관 HITL 검토, 통신 단절·재밍 스트레스 시험, AAR 재보정, 최종 통합까지 다중 센서 중복 항적 통합 시뮬레이터를 단계별로 구현한다. 최종 목표는 비행체 하나를 하나의 항적으로 정확히 보여 주어 지휘관의 인지 부하를 줄이고, 항적 통합·분리 같은 판단은 조작관이 근거와 함께 확정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/ops02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/ui/`, `src/hitl/` | 전처리, 모델, 대시보드, HITL 승인 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/ops02_data_spec.md`, `docs/ops02_hitl_guide.md`, `docs/ops02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서 등 실무/지휘관 문서 |
| **`configs/`**| `configs/ops02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/ops02_stress_test.py` | 500% 부하 시험 및 pytest 단위/회귀 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (추천):** 코드 셀 첫 줄에 `%%writefile src/models/ops02_baseline_model.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 센서별 오탐 가중치 반영 및 항적 통합 신뢰도 지수 산출 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

센서별 신뢰도 가중치와 기상 환경 요인을 반영하여 통합된 항적의 진위 여부 및 융합 신뢰도 지수를 계산하는 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **센서별 신뢰도 가중치와 기상 요인을 반영해 통합 항적의 진위를 나타내는 융합 신뢰도 지수와 High/Med/Low 등급을 산출한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **기상별 센서 신뢰도 가중치 매트릭스를 만들고 항적 수명, 스캔당 보고 수, 센서 종류 수, 가중 센서 지지도, 측정 정합도(NIS), EO/IR 단독 여부를 특징으로 산출하고 모델에 적용하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/verification/ops02_confidence_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 신뢰도 등급(High/Med/Low) 부여 기준의 군사적 타당성

#### 미션 PASS 조건

- 정량 기준: 오경보율(FAR) 40% 이상 감축, 신뢰도 스코어 산출 일관성 90% 이상


- 진행 기준: 오경보 필터링 전후 비교 지표 산출 및 통합 테스트 완료


- 검증 방법: ROC-AUC 곡선 분석 및 임계값 민감도 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""통합 항적마다 센서별 신뢰도 가중치(기상 반영), 항적 수명, 스캔당 보고 수, 센서 종류 수, 측정 정합도(NIS)를 특징으로 융합 신뢰도 지수와 High/Med/Low 등급을 산출해, 모든 확정 항적을 경보하는 방식 대비 오경보율 감축과 등급 일관성을 확인한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------------------------
# [교육용가상자료] 다중 센서(레이더·EO/IR·음향) 경보 생성기 (실제 장비 제원·배치와 무관)
# 가상 방어 지역(원점 기준 동/북 km)에 비행체 궤적을 만들고, 센서마다 '자기 위치 기준
# 거리·방위'와 '자기 시계 기준 시각'으로 경보를 낸다. 같은 비행체가 여러 센서에 동시에
# 잡히므로 경보가 중복되고, 센서마다 시계 오차·측정 오차·오경보가 다르다.
# ---------------------------------------------------------------------------
SENSORS = {
    "RDR-A": {"type": "radar", "pos": (0.0, 0.0), "sr": 0.08, "saz": 0.4, "pd": 0.95, "fa": 1.0, "max_r": 40.0, "clock": 0.0},
    "RDR-B": {"type": "radar", "pos": (16.0, -6.0), "sr": 0.10, "saz": 0.5, "pd": 0.93, "fa": 1.0, "max_r": 40.0, "clock": 0.42},
    "EOIR-C": {"type": "eoir", "pos": (5.0, 9.0), "sr": 0.40, "saz": 0.15, "pd": 0.88, "fa": 0.5, "max_r": 18.0, "clock": -0.65},
    "ACU-D": {"type": "acoustic", "pos": (10.0, 2.0), "sr": 0.60, "saz": 2.0, "pd": 0.80, "fa": 0.6, "max_r": 9.0, "clock": 0.93},
}
CENTER = np.array([8.0, 2.0])                         # 가상 방어 지역 중심
KINDS = {"uav": (0.03, 0.06), "heli": (0.04, 0.07), "fast": (0.15, 0.20)}   # 속도 범위(km/s, 추상값)
WEATHER = {"clear": {"eoir": 1.0, "acoustic": 1.0, "radar": 1.0, "fa": 1.0},
           "night": {"eoir": 0.9, "acoustic": 1.0, "radar": 1.0, "fa": 1.1},
           "fog": {"eoir": 0.55, "acoustic": 1.0, "radar": 1.0, "fa": 1.2},
           "rain": {"eoir": 0.7, "acoustic": 0.6, "radar": 0.97, "fa": 1.6}}


def _targets(rng, n, T, special):
    # [TODO : 학생 실습 5] 센서별 오탐 가중치 반영 및 항적 통합 신뢰도 지수 산출 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_scenario(seed, n=6, T=60, weather="clear", fa_mult=1.0, dropout=None, ghost=False, special=None,
                  noise_mult=None, drift=0.0):
    """시나리오 1개: 정답 궤적, 센서 경보(보고) 목록, 시계 동기화용 기준 신호(heartbeat)를 만든다.
    dropout=(센서, 시작, 끝): 통신 단절, ghost: 한 센서에만 보이는 거짓 항적(기만·스푸핑),
    noise_mult={센서: 배수}: 재밍으로 측정 잡음 증가, drift: 음향 센서 시계가 초당 drift초씩 흐트러짐."""
    rng = np.random.default_rng(seed)
    targets, kinds = _targets(rng, n, T, special)
    w = WEATHER[weather]
    reports, hb, mid = [], [], 0
    gpos = CENTER + np.array([-9.0, 7.0])
    for t in range(T):
        for sid, s in SENSORS.items():
            clock = s["clock"] + (drift * t if sid == "ACU-D" else 0.0)
            if t % 10 == 0:
                hb.append({"sensor": sid, "t_ref": float(t), "t_sensor": t + clock + rng.normal(0, 0.02)})
            if dropout and sid == dropout[0] and dropout[1] <= t < dropout[2]:
                continue
            nm = (noise_mult or {}).get(sid, 1.0)
            objs = [(tid, tr[t, :2]) for tid, tr in targets.items()]
            if ghost and sid == "EOIR-C":
                objs.append(("GHOST", gpos + 0.12 * t * np.array([0.6, -0.8])))
            for tid, xy in objs:
                d = xy - np.array(s["pos"])
                r = float(np.hypot(*d))
                if r > s["max_r"] or rng.random() > s["pd"] * w[s["type"]]:
                    continue
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(r + rng.normal(0, s["sr"] * nm), 3),
                                "az_deg": round(float((np.degrees(np.arctan2(d[0], d[1])) + rng.normal(0, s["saz"] * nm)) % 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": tid})
                mid += 1
            for _ in range(rng.poisson(s["fa"] * fa_mult * w["fa"] * (3 if nm > 1 else 1))):   # 오경보(클러터·반사)
                reports.append({"msg_id": f"M{mid:05d}", "sensor": sid, "stype": s["type"],
                                "t_sensor": round(t + clock + rng.normal(0, 0.02), 3),
                                "range_km": round(rng.uniform(0.5, s["max_r"]), 3), "az_deg": round(rng.uniform(0, 360), 3),
                                "sr": s["sr"] * nm, "saz": s["saz"] * nm, "truth": "FA"})
                mid += 1
    return {"targets": targets, "kinds": kinds, "reports": reports, "heartbeats": hb, "T": T, "weather": weather}
# ---------------------------------------------------------------------------

# ---- 시공간 전처리 공통 함수: 정제 -> 시각 동기화 -> 공통 좌표(ENU, km) 변환 ----
def estimate_offsets(heartbeats):
    """센서별 시계 오차 = 기준 신호(heartbeat) 대비 센서 시각 차이의 중앙값(잡음에 강함)."""
    # [TODO : 학생 실습 5] 센서별 오탐 가중치 반영 및 항적 통합 신뢰도 지수 산출 엔진 개발 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def clean(reports):
    """중복 메시지·결측·범위 밖(반사파 등 최대 탐지거리 초과) 경보를 걸러낸다. (정제 목록, 사유별 건수)"""
    seen, out, why = set(), [], {"중복": 0, "결측": 0, "범위 밖": 0}
    for r in reports:
        if r["msg_id"] in seen:
            why["중복"] += 1; continue
        seen.add(r["msg_id"])
        if any(r.get(k) is None or (isinstance(r.get(k), float) and np.isnan(r[k])) for k in ("t_sensor", "range_km", "az_deg")):
            why["결측"] += 1; continue
        if not 0 <= r["range_km"] <= SENSORS[r["sensor"]]["max_r"] * 1.1:
            why["범위 밖"] += 1; continue
        out.append(r)
    return out, why


def to_enu(r, offsets):
    """센서 국소 극좌표(거리·방위) -> 공통 ENU(동·북 km), 센서 시각 -> 기준 시각."""
    s, a = SENSORS[r["sensor"]], np.radians(r["az_deg"])
    return {**r, "t": r["t_sensor"] - offsets.get(r["sensor"], 0.0),
            "x": s["pos"][0] + r["range_km"] * np.sin(a), "y": s["pos"][1] + r["range_km"] * np.cos(a)}


def meas_cov(r):
    """거리·방위 오차를 ENU 공분산(2x2)으로 변환한다(1차 근사)."""
    a, rr = np.radians(r["az_deg"]), r["range_km"]
    J = np.array([[np.sin(a), rr * np.cos(a)], [np.cos(a), -rr * np.sin(a)]])
    return J @ np.diag([r["sr"] ** 2, np.radians(r["saz"]) ** 2]) @ J.T


def prepare(scn, sync=True):
    reps, _ = clean(scn["reports"])
    off = estimate_offsets(scn["heartbeats"]) if sync else {}
    return sorted((to_enu(r, off) for r in reps), key=lambda r: r["t"])

# ---- 확장 칼만 필터(EKF) 다중 센서 항적 추적기 공통 함수 ----
# 상태 x = [동(km), 북(km), 동속(km/s), 북속(km/s)], 측정 z = 센서 기준 [거리(km), 방위(rad)]
def ekf_predict(x, P, dt, q):
    F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    G = np.array([[dt ** 2 / 2, 0], [0, dt ** 2 / 2], [dt, 0], [0, dt]])
    return F @ x, F @ P @ F.T + q * G @ G.T


def h_polar(x, spos):
    dx, dy = x[0] - spos[0], x[1] - spos[1]
    r2 = max(dx * dx + dy * dy, 1e-9); r = np.sqrt(r2)
    return np.array([r, np.arctan2(dx, dy)]), np.array([[dx / r, dy / r, 0, 0], [dy / r2, -dx / r2, 0, 0]])


def innovation(x, P, rep):
    s = SENSORS[rep["sensor"]]
    zp, H = h_polar(x, s["pos"])
    y = np.array([rep["range_km"], np.radians(rep["az_deg"])]) - zp
    y[1] = (y[1] + np.pi) % (2 * np.pi) - np.pi                      # 방위 차이는 -pi~pi로 감싼다
    S = H @ P @ H.T + np.diag([rep["sr"] ** 2, np.radians(rep["saz"]) ** 2])
    return y, H, S, float(y @ np.linalg.solve(S, y))


class Tracker:
    """센서 경보를 스캔(1초) 단위로 받아 EKF 항적을 갱신한다. 같은 스캔 안에서 센서별로
    순차 갱신하므로, 여러 센서가 같은 비행체를 보면 하나의 항적으로 통합된다."""

    def __init__(self, gate=9.21, q=4e-4, confirm=(3, 5), max_miss=4, merge_gate=6.0):
        self.gate, self.q, self.confirm, self.max_miss, self.merge_gate = gate, q, confirm, max_miss, merge_gate
        self.tracks, self.dead, self.nid, self.n_merged = [], [], 0, 0

    def _new(self, rep, k):
        C = meas_cov(rep)
        P = np.zeros((4, 4)); P[:2, :2] = C; P[2, 2] = P[3, 3] = 0.25 ** 2
        self.tracks.append({"id": f"FT{self.nid:03d}", "x": np.array([rep["x"], rep["y"], 0, 0]), "P": P, "k": k,
                            "scans": [k], "sensors": {rep["sensor"]: 1}, "truths": [rep["truth"]], "nis": [],
                            "hist": [(k, rep["x"], rep["y"])], "confirmed": False, "miss": 0, "born": k, "upd": True})
        self.nid += 1

    def _assign(self, tracks, rs, sid):
        """게이트(마할라노비스 거리) 안에서 전역 최근접 할당(GNN). 할당된 보고서 번호 집합을 돌려준다."""
        if not tracks or not rs:
            return set()
        spos = SENSORS[sid]["pos"]
        Z = np.array([[r["range_km"], np.radians(r["az_deg"])] for r in rs])
        R = np.diag([rs[0]["sr"] ** 2, np.radians(rs[0]["saz"]) ** 2])
        cost = np.full((len(tracks), len(rs)), 1e6)
        for i, tr in enumerate(tracks):
            zp, H = h_polar(tr["x"], spos)
            Y = Z - zp
            Y[:, 1] = (Y[:, 1] + np.pi) % (2 * np.pi) - np.pi
            Si = np.linalg.inv(H @ tr["P"] @ H.T + R)
            d2 = np.einsum("ij,jk,ik->i", Y, Si, Y)
            cost[i] = np.where(d2 < self.gate, d2, 1e6)
        used = set()
        for i, j in zip(*linear_sum_assignment(cost)):
            if cost[i, j] < self.gate:
                tr, r = tracks[i], rs[j]
                y, H, S, d2 = innovation(tr["x"], tr["P"], r)
                K = tr["P"] @ H.T @ np.linalg.inv(S)
                tr["x"] = tr["x"] + K @ y
                tr["P"] = (np.eye(4) - K @ H) @ tr["P"]
                tr["nis"].append(d2); tr["truths"].append(r["truth"])
                tr["sensors"][sid] = tr["sensors"].get(sid, 0) + 1
                tr["upd"] = True; used.add(j)
        return used

    def _merge(self):
        """같은 비행체를 따라가는 중복 항적(위치·속도가 통계적으로 같음)을 오래된 항적으로 통합한다."""
        drop = set()
        for a in range(len(self.tracks)):
            for b in range(a + 1, len(self.tracks)):
                A, B = self.tracks[a], self.tracks[b]
                if A["id"] in drop or B["id"] in drop:
                    continue
                d = A["x"] - B["x"]
                d2 = float(d[:2] @ np.linalg.solve(A["P"][:2, :2] + B["P"][:2, :2], d[:2]))
                if d2 < self.merge_gate and np.hypot(d[2], d[3]) < 0.05:
                    keep, lose = (A, B) if len(A["scans"]) >= len(B["scans"]) else (B, A)
                    for sid, n in lose["sensors"].items():
                        keep["sensors"][sid] = keep["sensors"].get(sid, 0) + n
                    keep["truths"] += lose["truths"]; keep["scans"] = sorted(set(keep["scans"]) | set(lose["scans"]))
                    keep["merged"] = keep.get("merged", 0) + 1
                    drop.add(lose["id"])
        self.tracks = [t for t in self.tracks if t["id"] not in drop]
        self.n_merged += len(drop)

    def step(self, k, reps):
        for tr in self.tracks:                                         # 1) 예측
            tr["x"], tr["P"] = ekf_predict(tr["x"], tr["P"], k - tr["k"], self.q)
            tr["k"], tr["upd"] = k, False
        for sid in sorted({r["sensor"] for r in reps}, key=lambda s: SENSORS[s]["sr"]):   # 2) 정밀 센서부터 순차 갱신
            rs = [r for r in reps if r["sensor"] == sid]
            used = self._assign([t for t in self.tracks if t["confirmed"]], rs, sid)      # 확정 항적 우선
            rest = [r for j, r in enumerate(rs) if j not in used]
            used2 = self._assign([t for t in self.tracks if not t["confirmed"]], rest, sid)
            for j, r in enumerate(rest):
                if j not in used2:
                    self._new(r, k)
        if self.merge_gate > 0:
            self._merge()
        keep = []
        for tr in self.tracks:                                         # 3) 확정·삭제 관리
            if tr["upd"]:
                tr["miss"] = 0
                if tr["scans"][-1] != k:
                    tr["scans"].append(k)
                tr["hist"].append((k, tr["x"][0], tr["x"][1]))
            else:
                tr["miss"] += 1
            m, n = self.confirm
            if not tr["confirmed"] and sum(1 for s in tr["scans"] if s > k - n) >= m:
                tr["confirmed"] = True
            limit = self.max_miss if tr["confirmed"] else 1
            (keep if tr["miss"] <= limit else self.dead).append(tr)
        self.tracks = keep

    def all_tracks(self):
        return self.dead + self.tracks


def run_tracker(reps, **kw):
    trk = Tracker(**kw)
    by = {}
    for r in reps:
        by.setdefault(int(round(r["t"])), []).append(r)
    for k in range(min(by), max(by) + 1):
        trk.step(k, by.get(k, []))
    return trk.all_tracks()


def label_track(tr):
    vals, cnt = np.unique(tr["truths"], return_counts=True)
    return str(vals[np.argmax(cnt)]), float(cnt.max() / cnt.sum())


def evaluate(tracks, scn, min_reports=5):
    """정답 연결 평가: 재현율, 항적 분할(한 표적에 항적 2개 이상), 거짓 항적, 병합 오류, 위치 오차."""
    conf = [t for t in tracks if t["confirmed"]]
    for t in conf:
        t["label"], t["purity"] = label_track(t)
    cnt = pd.Series([r["truth"] for r in scn["reports"] if r["truth"].startswith("TGT")]).value_counts()
    truths = [k for k, v in cnt.items() if v >= min_reports]
    per = {g: [t for t in conf if t["label"] == g] for g in truths}
    found = [g for g in truths if per[g]]
    split = sum(max(0, len(v) - 1) for v in per.values())
    false = [t for t in conf if not t["label"].startswith("TGT")]
    merge = [t for t in conf if t["label"].startswith("TGT") and t["purity"] < 0.7]
    err = []
    for g in found:
        for t in per[g]:
            for k, x, y in t["hist"][3:]:
                if 0 <= k < scn["T"]:
                    err.append(np.hypot(x - scn["targets"][g][k, 0], y - scn["targets"][g][k, 1]))
    n = len(truths)
    acc = len(found) / max(n + split + len(false) + len(merge), 1)
    return {"정답표적": n, "탐지표적": len(found), "재현율": round(len(found) / max(n, 1), 3), "분할": split,
            "거짓항적": len(false), "병합오류": len(merge), "위치RMSE(km)": round(float(np.sqrt(np.mean(np.square(err)))) if err else np.nan, 3),
            "항적정확도": round(acc, 3)}

# 센서 신뢰도 가중치 매트릭스(기상별) [교육용가상자료]: 기본 신뢰도 x 기상 탐지 계수
BASE_W = {"radar": 1.0, "eoir": 0.8, "acoustic": 0.6}
WMAT = {w: {sid: round(BASE_W[s["type"]] * WEATHER[w][s["type"]] / WEATHER[w]["fa"], 2) for sid, s in SENSORS.items()} for w in WEATHER}


def track_features(trk, weather):
    rows = []
    for t in trk:
        if not t["confirmed"]:
            continue
        n = len(t["scans"])
        rows.append({"id": t["id"], "life": min(n, 10) / 10, "rps": len(t["truths"]) / n, "nsens": len(t["sensors"]),
                     "wsup": sum(WMAT[weather][s] * min(c / n, 1) for s, c in t["sensors"].items()),
                     "nis": float(np.median(t["nis"])) if t["nis"] else 9.0, "eoir_only": float(set(t["sensors"]) == {"EOIR-C"}),
                     "label": t["label"], "true": t["label"].startswith("TGT")})
    return pd.DataFrame(rows)


FEAT = ["life", "rps", "nsens", "wsup", "nis", "eoir_only"]


def dataset(seeds, fa=2.0, **kw):
    out = []
    for i, sd in enumerate(seeds):
        w = ["clear", "night", "fog", "rain"][i % 4]
        scn = make_scenario(sd, weather=w, fa_mult=fa, ghost=(i % 3 == 0), special=[None, "turn", "cross", "formation"][i % 4], **kw)
        reps = prepare(scn)
        for ts in (8, 15, 30):                                     # 실시간 판단 시점: 해당 시각까지의 보고만 사용
            trk = run_tracker([r for r in reps if r["t"] <= ts]); evaluate(trk, scn)
            df = track_features(trk, w); df["weather"] = w; df["scn"] = sd; df["ts"] = ts; out.append(df)
    return pd.concat(out, ignore_index=True)


print("[센서 신뢰도 가중치 매트릭스]")
print(pd.DataFrame(WMAT).T.to_string())
tr = dataset(range(500, 508))
te = dataset(range(600, 606), fa=4.0, noise_mult={"RDR-B": 3.0})          # 평가: 오경보 4배 + 레이더B 재밍(학습보다 가혹)
eng = LogisticRegression(max_iter=1000, C=1.0).fit(tr[FEAT], tr["true"])


def grade(F):
    s = eng.predict_proba(F[FEAT])[:, 1]
    g = np.where(s >= 0.8, "High", np.where(s >= 0.5, "Med", "Low"))
    return s, g


te["score"], te["grade"] = grade(te)
false_n = (~te["true"]).sum()
far_old = false_n / len(te)                                        # 기존: 확정 항적 전부 경보
alarm = te["grade"] == "High"
far_new = (~te["true"] & alarm).sum() / max(alarm.sum(), 1)
red = 1 - far_new / far_old
print(f"\n평가 항적(시점별) {len(te)}개(실제 {te['true'].sum()} / 거짓 {false_n}), 시나리오 6개 x 판단 시점 3개(8·15·30초), 기상 4종, 오경보 4배, 레이더B 재밍, 기만 항적 포함")
print(f"오경보율(경보 중 거짓 항적 비율): 기존 {far_old:.1%} -> High 경보 {far_new:.1%}, 감축 {red:.0%} (목표 40% 이상: {'달성' if red >= 0.4 else '미달'})")
print(f"실제 항적 High 비율 {(te['true'] & alarm).sum() / te['true'].sum():.1%}, 실제 항적 Low 판정 {(te['true'] & (te['grade'] == 'Low')).sum()}건, ROC-AUC {roc_auc_score(te['true'], te['score']):.3f}")
print("\n등급별 항적 수:")
print(pd.crosstab(te["grade"], te["true"].map({True: "실제", False: "거짓"})).reindex(["High", "Med", "Low"]).fillna(0).astype(int).to_string())

coef = pd.Series(eng.coef_[0], index=FEAT).round(2)
print("\n특징 가중치(로지스틱):", {k: float(v) for k, v in coef.items()})

# 일관성: 특징에 ±5% 측정 흔들림을 10회 주어도 등급이 유지되는 비율
rng = np.random.default_rng(7)
same = []
for _ in range(10):
    P = te.copy()
    for c in ["life", "rps", "wsup", "nis"]:
        P[c] = P[c] * rng.uniform(0.95, 1.05, len(P))
    same.append((grade(P)[1] == te["grade"]).mean())
cons = float(np.mean(same))
print(f"\n신뢰도 등급 일관성(특징 ±5% 흔들림 10회): {cons:.1%} (목표 90% 이상: {'달성' if cons >= 0.9 else '미달'})")
print("\n[임계값 민감도] High 기준  경보 수  경보 중 거짓")
for th in [0.6, 0.7, 0.8, 0.9]:
    a = te["score"] >= th
    print(f"               {th:.1f}      {a.sum():4d}   {(~te['true'] & a).sum() / max(a.sum(), 1):6.1%}")
print("등급 운용: High=통합 항적 경보 / Med=조작관 확인(미션 7) / Low=감시 유지(삭제하지 않음). 교전 여부는 교전수칙·지휘계통에 따른다.")

plt.figure(figsize=(6, 3.5))
plt.hist([te["score"][te["true"]], te["score"][~te["true"]]], bins=20, label=["true track", "false track"], color=["tab:blue", "tab:red"])
plt.axvline(0.8, c="k", ls="--"); plt.axvline(0.5, c="gray", ls=":"); plt.xlabel("fusion confidence"); plt.legend(); plt.title("Fusion confidence (synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[센서 신뢰도 가중치 매트릭스]
       RDR-A  RDR-B  EOIR-C  ACU-D
clear   1.00   1.00    0.80   0.60
night   0.91   0.91    0.65   0.55
fog     0.83   0.83    0.37   0.50
rain    0.61   0.61    0.35   0.22

평가 항적(시점별) 263개(실제 114 / 거짓 149), 시나리오 6개 x 판단 시점 3개(8·15·30초), 기상 4종, 오경보 4배, 레이더B 재밍, 기만 항적 포함
오경보율(경보 중 거짓 항적 비율): 기존 56.7% -> High 경보 2.6%, 감축 95% (목표 40% 이상: 달성)
실제 항적 High 비율 97.4%, 실제 항적 Low 판정 0건, ROC-AUC 0.998

등급별 항적 수:
true    거짓   실제
grade          
High     3  111
Med     18    3
Low    128    0

특징 가중치(로지스틱): {'life': -0.32, 'rps': 1.21, 'nsens': 1.32, 'wsup': 1.39, 'nis': -0.76, 'eoir_only': -0.61}

신뢰도 등급 일관성(특징 ±5% 흔들림 10회): 99.8% (목표 90% 이상: 달성)

[임계값 민감도] High 기준  경보 수  경보 중 거짓
               0.6       128    11.7%
               0.7       120     5.8%
               0.8       114     2.6%
               0.9       113     2.7%
등급 운용: High=통합 항적 경보 / Med=조작관 확인(미션 7) / Low=감시 유지(삭제하지 않음). 교전 여부는 교전수칙·지휘계통에 따른다.
```
</details>
